# BioDYM Material Flow Analysis - Scientific Notebook

A streamlined notebook for Material Flow Analysis using the BioDYM framework.

## Workflow
1. **Load Excel File** - Define input data
2. **Confirm Configuration** - Review loaded data and settings
3. **Run Calculation** - Execute MFA analysis
4. **Mass Balance Check** - Verify calculation accuracy
5. **Visualizations** - Display all available plots

---

## 1. Setup and Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown

In [ ]:
# Add BioDYM modules to path
src_path = os.path.join(os.getcwd(), 'src')
sys.path.insert(0, src_path)

In [ ]:
# Add ODYM framework to path
biodym_mfa_tool_dir = os.getcwd()
odym_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "ODYM-master_20241127", "odym", "modules"
)
sys.path.insert(0, odym_path)

In [ ]:
# Add bioDYM add-on to path
biodym_addon_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "bioDYM_add-on", "modules"
)
sys.path.insert(0, biodym_addon_path)

In [ ]:
# Import BioDYM modules
try:
    import config
    import data_loader
    import system_setup
    import utils
    from engine import solver
    import plotting
    print("✅ BioDYM modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

In [ ]:
# Set up plotting
plt.style.use('default')
print("📊 Plotting environment ready")

## 2. Define Input File

**Change this variable to your Excel file:**

In [ ]:
input_file = "data/01_input/250707_Template_CS1.xlsx"

In [ ]:
print(f"📁 Input file: {input_file}")

## 3. Load and Validate Data

In [ ]:
print("\n" + "="*60)
print("📊 LOADING AND VALIDATING DATA")
print("="*60)

In [ ]:
# Load Excel file
try:
    input_data = pd.read_excel(
        input_file,
        sheet_name=None,
        header=0,
        engine='openpyxl',
        na_values=['N.A.', 'NA', 'n/a']
    )
    print(f"✅ Excel file loaded: {len(input_data)} sheets")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    raise

In [ ]:
# Display sheet overview
print("\n📋 Sheet Overview:")
for sheet_name, df in input_data.items():
    print(f"   {sheet_name}: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Validate required sheets
required_sheets = [
    '1_1_Definition_Flows',
    '1_2_Data_Flows', 
    '2_1_Definition_Processes',
    '2_4_Initial_Stock',  # Correct sheet name
    '2_5_dynamic_tcs'
]

In [ ]:
missing_sheets = [sheet for sheet in required_sheets if sheet not in input_data.keys()]
if missing_sheets:
    print(f"\n⚠️ Missing required sheets: {missing_sheets}")
else:
    print("\n✅ All required sheets present")

## 4. Extract Configuration from Data

In [ ]:
print("\n" + "="*60)
print("⚙️ EXTRACTING CONFIGURATION")
print("="*60)

In [ ]:
# Extract time range from flow data
flow_data = input_data['1_2_Data_Flows']
years = sorted(flow_data['Year_Flow'].unique())
start_year = int(min(years))
end_year = int(max(years))

In [ ]:
print(f"📅 Time range: {start_year} - {end_year}")

In [ ]:
# Extract elements from flow data
elements = ['material', 'WC', 'DM', 'CC']  # Default elements
print(f"🧪 Elements: {elements}")

In [ ]:
# Check for Monte Carlo parameters
has_mc = '4_1_Uncertainty_Parameters' in input_data.keys()
print(f"🎲 Monte Carlo available: {'Yes' if has_mc else 'No'}")

In [ ]:
# Check for DSM parameters
has_dsm = '3_1_Definition_DSM' in input_data.keys()
print(f"📈 DSM available: {'Yes' if has_dsm else 'No'}")

In [ ]:
# Check for FOMP parameters
has_fomp = '3_2_Definition_FOMP' in input_data.keys()
print(f"🌱 FOMP available: {'Yes' if has_fomp else 'No'}")

## 5. Confirm Configuration

In [ ]:
print("\n" + "="*60)
print("✅ CONFIGURATION CONFIRMATION")
print("="*60)

In [ ]:
config_summary = f"""
**Analysis Configuration:**
- Input File: {input_file}
- Time Range: {start_year} - {end_year}
- Elements: {', '.join(elements)}
- Monte Carlo: {'Enabled' if has_mc else 'Disabled'}
- DSM: {'Enabled' if has_dsm else 'Disabled'}
- FOMP: {'Enabled' if has_fomp else 'Disabled'}
"""

In [ ]:
display(Markdown(config_summary))

## 6. Run MFA Calculation

In [ ]:
print("\n" + "="*60)
print("🚀 RUNNING MFA CALCULATION")
print("="*60)

In [ ]:
# 1. Setup model scope
print("📋 Setting up model scope...")
try:
    model_classification, index_table = system_setup.define_model_scope(
        start_year, end_year, elements
    )
    print("✅ Model scope defined")
except Exception as e:
    print(f"❌ Error setting up model scope: {e}")
    raise

In [ ]:
# 2. Initialize MFA system
print("🔧 Initializing MFA system...")
try:
    mfa_system_base = system_setup.initialize_mfa_system(
        model_classification, index_table
    )
    print("✅ MFA system initialized")
except Exception as e:
    print(f"❌ Error initializing MFA system: {e}")
    raise

In [ ]:
# 3. Load and define processes
print("📊 Loading processes and data...")
try:
    mfa_system_base, all_excel_data = system_setup.load_and_define_processes(
        mfa_system_base, input_file, data_loader
    )
    print("✅ Processes and data loaded")
except Exception as e:
    print(f"❌ Error loading processes: {e}")
    raise

In [ ]:
# 4. Load parameters
print("⚙️ Loading parameters...")
try:
    dsm_params = data_loader.load_dsm_parameters(all_excel_data)
    fomp_params = data_loader.load_fomp_parameters(all_excel_data)
    uncertainty_params = data_loader.load_uncertainty_definitions(all_excel_data)
    print("✅ Parameters loaded")
except Exception as e:
    print(f"❌ Error loading parameters: {e}")
    raise

In [ ]:
# 5. Define flows and parameters
print("🔗 Defining flows and parameters...")
try:
    mfa_system_configured, _ = system_setup.define_flows_and_parameters(
        mfa_system_base, all_excel_data
    )
    print(f"✅ System configured: {len(mfa_system_configured.ProcessList)} processes, "
          f"{len(mfa_system_configured.FlowDict)} flows, {len(mfa_system_configured.StockDict)} stocks")
except Exception as e:
    print(f"❌ Error defining flows and parameters: {e}")
    raise

In [ ]:
# 6. Run calculation
print("🧮 Running calculation...")
try:
    mfa_system_with_results, dsm_details = solver.run_mfa_calculation(
        mfa_system_configured, dsm_params, fomp_params, config
    )
    print("✅ Calculation completed successfully!")
except Exception as e:
    print(f"❌ Calculation error: {e}")
    import traceback
    traceback.print_exc()
    raise

## 7. Mass Balance Check

In [ ]:
print("\n" + "="*60)
print("⚖️ MASS BALANCE VERIFICATION")
print("="*60)

In [ ]:
# Calculate mass balance errors
mass_balance_errors = []
for process in mfa_system_with_results.ProcessList:
    if hasattr(process, 'MassBalance') and process.MassBalance is not None:
        for year_idx, year in enumerate(range(start_year, end_year + 1)):
            for element_idx, element in enumerate(elements):
                error = process.MassBalance[year_idx, element_idx]
                if abs(error) > 1e-6:  # Significant error threshold
                    mass_balance_errors.append({
                        'Process': process.Name,
                        'Year': year,
                        'Element': element,
                        'Error': error
                    })

In [ ]:
if mass_balance_errors:
    print("⚠️ Mass balance errors detected:")
    error_df = pd.DataFrame(mass_balance_errors)
    display(error_df)
else:
    print("✅ All mass balances within acceptable limits")

## 8. Results Overview

In [ ]:
print("\n" + "="*60)
print("📈 RESULTS OVERVIEW")
print("="*60)

In [ ]:
# Display final stock values
print("\n📊 Final Stock Values (Year {end_year}):")
final_stocks = []
for stock_name, stock in mfa_system_with_results.StockDict.items():
    if stock_name.startswith('S_'):  # Absolute stocks only
        final_value = stock.Values[-1, 0]  # Material dimension, final year
        final_stocks.append({
            'Stock': stock_name,
            'Final Value (Mg)': final_value
        })

In [ ]:
if final_stocks:
    stocks_df = pd.DataFrame(final_stocks)
    display(stocks_df)

In [ ]:
# Display flow summary
print("\n🔄 Flow Summary:")
flow_summary = []
for flow_id, flow in mfa_system_with_results.FlowDict.items():
    avg_flow = np.mean(flow.Values[:, 0])  # Average material flow
    flow_summary.append({
        'Flow ID': flow_id,
        'From': flow.P_Start,
        'To': flow.P_End,
        'Avg Flow (Mg/year)': avg_flow
    })

In [ ]:
if flow_summary:
    flows_df = pd.DataFrame(flow_summary)
    display(flows_df.head(10))  # Show first 10 flows

## 9. Visualizations

In [ ]:
print("\n" + "="*60)
print("📊 VISUALIZATIONS")
print("="*60)

============================================================================
3.1 System Overview - Sankey Diagram
============================================================================

In [ ]:
print("\n" + "-"*40)
print("3.1 SYSTEM OVERVIEW - SANKEY DIAGRAM")
print("-"*40)

In [ ]:
def create_sankey_diagram(mfa_system, start_year, end_year, elements):
    """
    Create an interactive Sankey diagram from MFA system data.
    
    Args:
        mfa_system: MFA system with results
        start_year: Start year of analysis
        end_year: End year of analysis
        elements: List of elements to analyze
    
    Returns:
        plotly.graph_objects.Figure: Interactive Sankey diagram
    """
    print("🔗 Creating Sankey diagram...")
    
    # Collect all processes and flows
    processes = []
    flows = []
    
    # Add all processes as nodes
    for process in mfa_system.ProcessList:
        processes.append(process.Name)
    
    # Create flow data for Sankey
    source_nodes = []
    target_nodes = []
    flow_values = []
    flow_labels = []
    
    for flow_id, flow in mfa_system.FlowDict.items():
        # Calculate average flow over time period
        avg_flow = np.mean(flow.Values[:, 0])  # Material dimension
        
        if avg_flow > 0:  # Only include positive flows
            source_nodes.append(flow.P_Start)
            target_nodes.append(flow.P_End)
            flow_values.append(avg_flow)
            flow_labels.append(f"{flow_id}: {avg_flow:.1f} Mg/year")
    
    # Create node labels (unique processes)
    all_nodes = list(set(source_nodes + target_nodes))
    node_labels = all_nodes
    
    # Create node indices for Sankey
    node_to_index = {node: idx for idx, node in enumerate(all_nodes)}
    
    # Convert process names to indices
    source_indices = [node_to_index[node] for node in source_nodes]
    target_indices = [node_to_index[node] for node in target_nodes]
    
    # Create color scheme (can be customized later)
    colors = px.colors.qualitative.Set3[:len(all_nodes)]
    
    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=colors
        ),
        link=dict(
            source=source_indices,
            target=target_indices,
            value=flow_values,
            label=flow_labels,
            color=['rgba(0,0,0,0.3)'] * len(flow_values)  # Semi-transparent links
        )
    )])
    
    # Update layout
    fig.update_layout(
        title_text=f"Material Flow Sankey Diagram ({start_year}-{end_year})",
        font_size=10,
        height=600,
        width=1000
    )
    
    print(f"✅ Sankey diagram created with {len(all_nodes)} nodes and {len(flow_values)} flows")
    return fig

In [ ]:
# Create and display Sankey diagram
try:
    sankey_fig = create_sankey_diagram(mfa_system_with_results, start_year, end_year, elements)
    sankey_fig.show()
except Exception as e:
    print(f"⚠️ Could not create Sankey diagram: {e}")
    import traceback
    traceback.print_exc()

============================================================================
3.2 System Overview - Stock Overview
============================================================================

In [ ]:
print("\n" + "-"*40)
print("3.2 SYSTEM OVERVIEW - STOCK OVERVIEW")
print("-"*40)

In [ ]:
def create_stock_overview(mfa_system, end_year, elements):
    """
    Create stock overview visualizations.
    
    Args:
        mfa_system: MFA system with results
        end_year: End year for final stock values
        elements: List of elements to analyze
    
    Returns:
        plotly.graph_objects.Figure: Stock overview figure
    """
    print("📊 Creating stock overview...")
    
    # Collect final stock values for each element
    stock_data = []
    
    for stock_name, stock in mfa_system.StockDict.items():
        if stock_name.startswith('S_'):  # Absolute stocks only
            for element_idx, element in enumerate(elements):
                if element_idx < stock.Values.shape[1]:  # Check if element exists
                    final_value = stock.Values[-1, element_idx]  # Final year, element dimension
                    stock_data.append({
                        'Stock': stock_name,
                        'Element': element,
                        'Value': final_value
                    })
    
    if not stock_data:
        print("⚠️ No stock data available")
        return None
    
    # Create subplots for each element
    element_list = list(set([d['Element'] for d in stock_data]))
    fig = make_subplots(
        rows=len(element_list), cols=1,
        subplot_titles=[f"Final Stock Values - {element}" for element in element_list],
        vertical_spacing=0.1
    )
    
    for idx, element in enumerate(element_list):
        element_data = [d for d in stock_data if d['Element'] == element]
        
        if element_data:
            stocks = [d['Stock'] for d in element_data]
            values = [d['Value'] for d in element_data]
            
            fig.add_trace(
                go.Bar(
                    x=stocks,
                    y=values,
                    name=element,
                    showlegend=False
                ),
                row=idx+1, col=1
            )
    
    fig.update_layout(
        title_text=f"Stock Overview - Final Values (Year {end_year})",
        height=200 * len(element_list),
        width=800,
        showlegend=False
    )
    
    print(f"✅ Stock overview created for {len(element_list)} elements")
    return fig

In [ ]:
# Create and display stock overview
try:
    stock_fig = create_stock_overview(mfa_system_with_results, end_year, elements)
    if stock_fig:
        stock_fig.show()
except Exception as e:
    print(f"⚠️ Could not create stock overview: {e}")

============================================================================
3.3 System Overview - Flow Overview
============================================================================

In [ ]:
print("\n" + "-"*40)
print("3.3 SYSTEM OVERVIEW - FLOW OVERVIEW")
print("-"*40)

In [ ]:
def create_flow_overview(mfa_system):
    """
    Create flow overview visualization.
    
    Args:
        mfa_system: MFA system with results
    
    Returns:
        plotly.graph_objects.Figure: Flow overview figure
    """
    print("🔄 Creating flow overview...")
    
    # Collect flow data
    flow_data = []
    
    for flow_id, flow in mfa_system.FlowDict.items():
        avg_flow = np.mean(flow.Values[:, 0])  # Average material flow
        if avg_flow > 0:  # Only significant flows
            flow_data.append({
                'Flow ID': flow_id,
                'From': flow.P_Start,
                'To': flow.P_End,
                'Avg Flow (Mg/year)': avg_flow
            })
    
    if not flow_data:
        print("⚠️ No flow data available")
        return None
    
    # Sort by flow magnitude
    flow_data.sort(key=lambda x: x['Avg Flow (Mg/year)'], reverse=True)
    
    # Create bar chart of top flows
    top_flows = flow_data[:15]  # Show top 15 flows
    
    fig = go.Figure()
    
    flow_labels = [f"{f['From']} → {f['To']}" for f in top_flows]
    flow_values = [f['Avg Flow (Mg/year)'] for f in top_flows]
    
    fig.add_trace(go.Bar(
        x=flow_labels,
        y=flow_values,
        text=[f"{v:.1f}" for v in flow_values],
        textposition='auto',
    ))
    
    fig.update_layout(
        title_text="Top Material Flows (Average)",
        xaxis_title="Flow (From → To)",
        yaxis_title="Average Flow (Mg/year)",
        height=500,
        width=800,
        xaxis_tickangle=-45
    )
    
    print(f"✅ Flow overview created with {len(top_flows)} top flows")
    return fig

In [ ]:
# Create and display flow overview
try:
    flow_fig = create_flow_overview(mfa_system_with_results)
    if flow_fig:
        flow_fig.show()
except Exception as e:
    print(f"⚠️ Could not create flow overview: {e}")

============================================================================
3.4 Individual Process Analysis (Placeholder)
============================================================================

In [ ]:
print("\n" + "-"*40)
print("3.4 INDIVIDUAL PROCESS ANALYSIS")
print("-"*40)

In [ ]:
print("🔧 Process analysis section - to be implemented in next step")
print("This will include:")
print("- Process selector widget")
print("- Regular process visualizations")
print("- DSM process visualizations")
print("- FOMP process visualizations")

============================================================================
3.5 Flow Analysis (Placeholder)
============================================================================

In [ ]:
print("\n" + "-"*40)
print("3.5 FLOW ANALYSIS")
print("-"*40)

In [ ]:
print("📈 Flow analysis section - to be implemented in next step")
print("This will include:")
print("- Flow time series")
print("- Flow correlations")
print("- Transfer coefficients")

============================================================================
3.6 Monte Carlo Analysis (Placeholder)
============================================================================

In [ ]:
print("\n" + "-"*40)
print("3.6 MONTE CARLO ANALYSIS")
print("-"*40)

In [ ]:
print("🎲 Monte Carlo analysis section - to be implemented in next step")
print("This will include:")
print("- Uncertainty quantification")
print("- Sensitivity analysis")
print("- Process-specific MC results")

## 10. Export Results

In [ ]:
print("\n" + "="*60)
print("💾 EXPORTING RESULTS")
print("="*60)

In [ ]:
# Export to Excel
output_file = "data/02_output/results_scientific.xlsx"
try:
    utils.export_results_to_excel(mfa_system_with_results, output_file)
    print(f"✅ Results exported to: {output_file}")
except Exception as e:
    print(f"⚠️ Export error: {e}")

In [ ]:
# Export configuration summary
config_file = output_file.replace('.xlsx', '_config.xlsx')
try:
    config_summary = pd.DataFrame([{
        'Input File': input_file,
        'Start Year': start_year,
        'End Year': end_year,
        'Elements': ', '.join(elements),
        'Monte Carlo': has_mc,
        'DSM': has_dsm,
        'FOMP': has_fomp
    }])
    config_summary.to_excel(config_file, index=False)
    print(f"✅ Configuration exported to: {config_file}")
except Exception as e:
    print(f"⚠️ Config export error: {e}")

## 11. Summary

In [ ]:
print("\n" + "="*60)
print("🎉 ANALYSIS COMPLETE")
print("="*60)

In [ ]:
summary = f"""
**Analysis Summary:**
- ✅ Input file processed successfully
- ✅ Configuration extracted automatically
- ✅ MFA calculation completed
- ✅ Mass balance verified
- ✅ Visualizations generated
- ✅ Results exported

**Key Results:**
- Time period: {start_year} - {end_year}
- Processes analyzed: {len(mfa_system_with_results.ProcessList)}
- Flows tracked: {len(mfa_system_with_results.FlowDict)}
- Stocks modeled: {len(mfa_system_with_results.StockDict)}
- Mass balance errors: {len(mass_balance_errors)}

**Files Generated:**
- Main results: {output_file}
- Configuration: {config_file}
"""

In [ ]:
display(Markdown(summary))

In [ ]:
print("\n📊 Analysis completed successfully!") 